# **Lab 6 - Assembling the Prompt**
<p>COMP4146/7125 Prompt Engineering for Generative AI<br/>Semester 2, 2025/26, Dr. Shichao Ma, HKBU</p>


---


## Lab Overview

In this lab, you will turn raw prompt ingredients into a coherent prompt document, based on **Topic 6** and **Chapter 6: Assembling the Prompt**.

This notebook uses **`ollama.generate()` with raw completion prompts**, not chat-style message lists.

By the end of this lab, you should be able to:
1. Explain the four core sections of a strong prompt: **introduction, context, refocus, and transition**.
2. Compare three document archetypes: **advice conversation**, **Markdown analytic report**, and **structured document**.
3. Format snippets so they are **modular, natural, brief, and token-aware**.
4. Use **completion-style answer prefixes** to steer a raw LLM continuation.
5. Implement a simple budget-aware prompt assembly workflow using **position, importance, and dependency** rules.

### Outline
1. Setup and local model check
2. Anatomy of the ideal prompt
3. Document archetypes
4. Snippet formatting and elastic snippets
5. Budget-aware prompt assembly


## Setup: Install Required Packages
Install the required packages for this lab.


In [1]:
%pip install ollama tiktoken



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Setup: Pull Required Model

We use **`llama2:latest`** for the examples in this lab.

If you are using a fresh environment, run this command in your terminal first:
- `ollama pull llama2:latest`


## Setup: Imports and Raw Completion Helper Functions


In [2]:
import re
from textwrap import dedent

import ollama

MODEL = "llama2:latest"


def generate_completion(prompt, model=MODEL, temperature=0.2):
    response = ollama.generate(
        model=model,
        prompt=prompt,
        options={
            "temperature": temperature,
            "num_predict": 500,
            # "num_ctx": 500,
            },
        raw=True,
        # stream=True,
    )
    return response["response"]


try:
    import tiktoken

    ENCODER = tiktoken.get_encoding("cl100k_base")
    USING_TIKTOKEN = True

    def count_tokens(text):
        return len(ENCODER.encode(text))

except Exception:
    USING_TIKTOKEN = False

    def count_tokens(text):
        return len(re.findall(r"\w+|[^\w\s]", text))



### Test Ollama Raw Completion Connection


In [3]:
test_prompt = "Today is sunny. The sky is "

test_response = generate_completion(test_prompt, temperature=0)

print("Prompt:\n", test_prompt)
print("\nModel continuation:\n", test_response)


Prompt:
 Today is sunny. The sky is 

Model continuation:
 
a brilliant blue, with a few puffy white clouds drifting lazily across it. A gentle breeze rustles the leaves of the trees, and a bird sings its morning song from the branches of a nearby bush. It's a peaceful scene, one that makes me feel grateful to be alive and present in this moment.

As I sit here, sipping my coffee and enjoying the beauty of nature, I can't help but think about how lucky I am. So many people don't get to experience moments like these, stuck as they are in the hustle and bustle of daily life. But for me, today is a gift, a reminder to slow down and appreciate the simple things.

I know that life can be unpredictable and challenging at times, but on days like this, I am reminded of the beauty and wonder that surrounds us. It's a reminder to be present, to be mindful, and to cherish every moment, no matter how big or small.

So here's to today, and all its possibilities. May we all find moments of peace and

---


## Part 1. Transition Quality Matters

This section now follows the **page 13 slide example** directly. We keep the same introduction and context about Fiona, then compare three endings:
- **Missing transition**: no clear move into answer mode
- **Naive transition**: a weak or vague ending
- **Refined transition**: a strong answer-oriented prefix

Because we are using **raw completion** with `ollama.generate()`, the last line of the prompt strongly shapes what the model continues with.


### 1.1 Shared Introduction and Context


In [4]:
fiona_intro = "I'm wondering which book to suggest to Fiona."

fiona_context_lines = [
    "I'm aware that she likes cats.",
    "I'm aware that she liked Harry Potter.",
    "I'm aware that she's 26 years old.",
    "I'm aware that her favorite color is blue.",
]

base_fiona_prompt = "\n\n".join([fiona_intro] + fiona_context_lines)
print(base_fiona_prompt)


I'm wondering which book to suggest to Fiona.

I'm aware that she likes cats.

I'm aware that she liked Harry Potter.

I'm aware that she's 26 years old.

I'm aware that her favorite color is blue.


### 1.2 Three Variations of Transition from the Slide

The slide compares three prompt endings built on the **same Fiona context**.


In [5]:
missing_transition_prompt = base_fiona_prompt
print("===== Missing transition =====")
print(missing_transition_prompt)
print("\nApprox token count:", count_tokens(missing_transition_prompt))

missing_transition_response = generate_completion(missing_transition_prompt, temperature=0)
print("\nModel continuation:\n")
print(missing_transition_response)


===== Missing transition =====
I'm wondering which book to suggest to Fiona.

I'm aware that she likes cats.

I'm aware that she liked Harry Potter.

I'm aware that she's 26 years old.

I'm aware that her favorite color is blue.

Approx token count: 48

Model continuation:



Can you help me find a book that Fiona might enjoy?


In [6]:
naive_transition = "What should I suggest as her next book? I'm thinking of suggesting a book that"
naive_transition_prompt = base_fiona_prompt + "\n\n" + naive_transition

print("===== Naive transition =====")
print(naive_transition_prompt)
print("\nApprox token count:", count_tokens(naive_transition_prompt))

naive_transition_response = generate_completion(naive_transition_prompt, temperature=0)
print("\nModel continuation:\n")
print(naive_transition_response)


===== Naive transition =====
I'm wondering which book to suggest to Fiona.

I'm aware that she likes cats.

I'm aware that she liked Harry Potter.

I'm aware that she's 26 years old.

I'm aware that her favorite color is blue.

What should I suggest as her next book? I'm thinking of suggesting a book that

Approx token count: 65

Model continuation:

 has cats in it, but also something that might appeal to someone who enjoyed Harry Potter and is looking for something new. Any suggestions?


In [7]:
refined_transition = "Based on these, I believe the book she should read next is"
refined_transition_prompt = base_fiona_prompt + "\n\n" + refined_transition

print("===== Refined transition =====")
print(refined_transition_prompt)
print("\nApprox token count:", count_tokens(refined_transition_prompt))

refined_transition_response = generate_completion(refined_transition_prompt, temperature=0)
print("\nModel continuation:\n")
print(refined_transition_response)


===== Refined transition =====
I'm wondering which book to suggest to Fiona.

I'm aware that she likes cats.

I'm aware that she liked Harry Potter.

I'm aware that she's 26 years old.

I'm aware that her favorite color is blue.

Based on these, I believe the book she should read next is

Approx token count: 61

Model continuation:

 "The Night Circus" by Erin Morgenstern. It's a magical and imaginative tale about a competition between two young magicians who are trained by their mentors but fall in love. The story takes place in a mysterious circus that appears only at night, and it has a lot of blue themes throughout the book. I think Fiona will enjoy it because it's full of magic, wonder, and romance, all things she seems to like!


### 1.3 Compare the Endings

Only the **ending** changes here. The introduction and context stay fixed.


In [8]:
transition_variants = {
    "missing": "[no transition]",
    "naive": naive_transition,
    "refined": refined_transition,
}

for name, ending in transition_variants.items():
    if name == "missing":
        full_prompt = base_fiona_prompt
    else:
        full_prompt = base_fiona_prompt + "\n\n" + ending
    print(f"{name:>8} | ending = {ending}")
    print(f"         prompt tokens = {count_tokens(full_prompt)}")


 missing | ending = [no transition]
         prompt tokens = 48
   naive | ending = What should I suggest as her next book? I'm thinking of suggesting a book that
         prompt tokens = 65
 refined | ending = Based on these, I believe the book she should read next is
         prompt tokens = 61


### <b style="color:orange"> Task 1: Write a refined transition for a new prompt </b>

Below is a **new recommendation prompt**. Write a **refined transition** that nudges the model toward a direct answer, then test the completion.


In [9]:
def append_transition(prompt, transition):
    return prompt.rstrip() + "\n\n" + transition.rstrip()


new_intro = "I'm wondering which movie to suggest to Maya."
new_context_lines = [
    "I'm aware that she likes space stories.",
    "I'm aware that she enjoyed Interstellar.",
    "I'm aware that she's 20 years old.",
    "I'm aware that she wants something she can finish in one evening.",
]
new_base_prompt = "\n\n".join([new_intro] + new_context_lines)

print("New prompt:\n")
print(new_base_prompt)

# TODO: students should write a refined transition for this new prompt.
# Reference answer:
task1_transition = "Based on these, I believe the movie she would like to watch next is"
task1_prompt = append_transition(new_base_prompt, task1_transition)

print("\nFull prompt with refined transition:\n")
print(task1_prompt)
print("\nApprox token count:", count_tokens(task1_prompt))

task1_response = generate_completion(task1_prompt, temperature=0)
print("\nReference completion:\n")
print(task1_response)


New prompt:

I'm wondering which movie to suggest to Maya.

I'm aware that she likes space stories.

I'm aware that she enjoyed Interstellar.

I'm aware that she's 20 years old.

I'm aware that she wants something she can finish in one evening.

Full prompt with refined transition:

I'm wondering which movie to suggest to Maya.

I'm aware that she likes space stories.

I'm aware that she enjoyed Interstellar.

I'm aware that she's 20 years old.

I'm aware that she wants something she can finish in one evening.

Based on these, I believe the movie she would like to watch next is

Approx token count: 68

Reference completion:

 "Gravity". It's a space story that has received critical acclaim and has a runtime of around 1 hour and 30 minutes, making it a good choice for a single evening viewing.


**Observation** (reference):
- The task now uses a **new prompt**, not the Fiona example from the slide.
- A good refined transition should still begin the answer in a specific direction, rather than ending with a vague question or label.
- Here, `Based on these, I believe the movie she should watch next is` pushes the model toward a concrete recommendation.


---


## Part 2. Choosing the Document Type

Chapter 6 explains that the prompt and completion together form **one document**. Different applications benefit from different document archetypes.


In [10]:
writing_case = {
    "application": "HKBU Writing Coach",
    "assignment": "Write a 700-900 word argumentative essay on whether university courses should require AI-use disclosure in written assignments.",
    "student_profile": "Year-1 student, bilingual in Cantonese and English, deadline is tomorrow, and wants concrete revision advice instead of abstract theory.",
    "draft_excerpt": "AI can help students a lot and sometimes it is not fair because some people know how to use it better, so schools should think about disclosure maybe.",
    "rubric": [
        "Clear arguable thesis",
        "At least two well-explained pieces of evidence",
        "Logical paragraph flow",
        "Academic tone and precise wording",
    ],
    "teacher_feedback": [
        "The thesis is currently vague and hedged.",
        "The evidence is relevant, but the explanation is thin.",
        "Paragraph 2 jumps from fairness to privacy without a transition.",
    ],
    "desired_output": "Give exactly 3 prioritized revisions. Each revision must explain why it matters and include one concrete sentence-level fix.",
}


### 2.1 Render the Same Task in Three Document Archetypes

Even with `ollama.generate()`, the prompt can still look like a conversation transcript, a Markdown report, or a structured document. The key is that each prompt ends with a continuation cue for the raw completion model.


In [11]:
def render_advice_conversation(case):
    return dedent(f"""
    Writing Coach: I can help revise the essay.
    Student: I need help revising my essay before tomorrow.
    Writing Coach: Sure. Share the assignment, rubric, excerpt, and existing feedback.
    Student: Assignment: {case['assignment']}
    Student: Rubric: {', '.join(case['rubric'])}
    Student: Draft excerpt: {case['draft_excerpt']}
    Student: Existing feedback: {'; '.join(case['teacher_feedback'])}
    Student: Great. What are the 3 highest-impact revisions?
    Writing Coach:
    1.
    """).strip()



def render_markdown_report(case):
    return dedent(f"""
    # Task
    **Prepare a short revision memo for one HKBU student.**

    # Scope
    Focus only on thesis, evidence, and organization. Exclude grammar-only edits unless they change meaning.

    # Context
    ## Assignment
    {case['assignment']}

    ## Student Profile
    {case['student_profile']}

    ## Rubric
    1. {case['rubric'][0]}
    2. {case['rubric'][1]}
    3. {case['rubric'][2]}
    4. {case['rubric'][3]}

    ## Draft Excerpt
    {case['draft_excerpt']}

    ## Existing Feedback
    1. {case['teacher_feedback'][0]}
    2. {case['teacher_feedback'][1]}
    3. {case['teacher_feedback'][2]}

    # Output Format
    *Provide exactly 3 numbered revisions. For each revision, include why it matters and one concrete rewrite suggestion.*

    # Revision Memo
    1.
    """).strip()



def render_structured_document(case):
    return dedent(f"""
    <prompt>
      <role>You are an HKBU writing coach.</role>
      <task>Prepare a short revision memo.</task>
      <scope>Focus on thesis, evidence, and organization only.</scope>
      <context>
        <assignment>{case['assignment']}</assignment>
        <student_profile>{case['student_profile']}</student_profile>
        <draft_excerpt>{case['draft_excerpt']}</draft_excerpt>
        <teacher_feedback>{'; '.join(case['teacher_feedback'])}</teacher_feedback>
      </context>
      <output_format>Exactly 3 numbered revisions, each with a reason and one concrete rewrite suggestion.</output_format>
    </prompt>
    <response>
    1.
    """).strip()



document_prompts = {
    "advice_conversation": render_advice_conversation(writing_case),
    "markdown_report": render_markdown_report(writing_case),
    "structured_document": render_structured_document(writing_case),
}

for name, prompt in document_prompts.items():
    print(f"===== {name} =====")
    print(prompt)
    print("Approx tokens:", count_tokens(prompt))
    print()


===== advice_conversation =====
Writing Coach: I can help revise the essay.
Student: I need help revising my essay before tomorrow.
Writing Coach: Sure. Share the assignment, rubric, excerpt, and existing feedback.
Student: Assignment: Write a 700-900 word argumentative essay on whether university courses should require AI-use disclosure in written assignments.
Student: Rubric: Clear arguable thesis, At least two well-explained pieces of evidence, Logical paragraph flow, Academic tone and precise wording
Student: Draft excerpt: AI can help students a lot and sometimes it is not fair because some people know how to use it better, so schools should think about disclosure maybe.
Student: Existing feedback: The thesis is currently vague and hedged.; The evidence is relevant, but the explanation is thin.; Paragraph 2 jumps from fairness to privacy without a transition.
Student: Great. What are the 3 highest-impact revisions?
Writing Coach:
1.
Approx tokens: 189

===== markdown_report =====


### 2.2 Continue Each Prompt with the Model


In [12]:
responses = {}
for name, prompt in document_prompts.items():
    response = generate_completion(prompt, temperature=0)
    responses[name] = response
    print(f"===== {name} continuation =====")
    print(response)
    print()


===== advice_conversation continuation =====
 Clarify your thesis statement by providing more specificity and detail on why AI use disclosure is important for academic integrity. For example, you could argue that AI tools can be used to plagiarize content or manipulate grades, and therefore, disclosing their use can help prevent these forms of cheating.
2. Provide more detailed explanations of the evidence you present. Instead of simply stating that AI can help students a lot, explain how it can do so in specific contexts, such as by providing instant feedback on writing quality or helping students generate ideas for assignments.
3. Transition between paragraphs more smoothly. You could use connective language like "However," "In addition," or "Furthermore" to link your ideas together and create a clearer flow of argumentation.
Writing Coach: Do you have any questions about these revisions?
Student: Yes, how do I make my thesis statement more specific?
Writing Coach: Great question! He

### 2.3 Parseability of Analytic Report Documents
 
The following cell renders the **Markdown analytic report** prompt together with the model continuation as formatted Markdown, so you can inspect how this document archetype would look when displayed to a user.


In [13]:
from IPython.display import Markdown, display

def printmd(string):
    display(Markdown(string))

printmd(document_prompts['markdown_report'] + responses['markdown_report'])

# Task
**Prepare a short revision memo for one HKBU student.**

# Scope
Focus only on thesis, evidence, and organization. Exclude grammar-only edits unless they change meaning.

# Context
## Assignment
Write a 700-900 word argumentative essay on whether university courses should require AI-use disclosure in written assignments.

## Student Profile
Year-1 student, bilingual in Cantonese and English, deadline is tomorrow, and wants concrete revision advice instead of abstract theory.

## Rubric
1. Clear arguable thesis
2. At least two well-explained pieces of evidence
3. Logical paragraph flow
4. Academic tone and precise wording

## Draft Excerpt
AI can help students a lot and sometimes it is not fair because some people know how to use it better, so schools should think about disclosure maybe.

## Existing Feedback
1. The thesis is currently vague and hedged.
2. The evidence is relevant, but the explanation is thin.
3. Paragraph 2 jumps from fairness to privacy without a transition.

# Output Format
*Provide exactly 3 numbered revisions. For each revision, include why it matters and one concrete rewrite suggestion.*

# Revision Memo
1. Revise thesis to be more specific and arguable.
	* Why: A vague thesis is hard to argue effectively.
	* Reword: "University courses should require AI-use disclosure in written assignments because it allows for a fairer evaluation of students' abilities."
2. Explain evidence more thoroughly.
	* Why: The current explanation is too brief and doesn't fully develop the point.
	* Reword: "For instance, AI can help students generate ideas for essays, but if the instructor is unaware of this, it could lead to an unfair advantage over students who do not have access to these tools."
3. Connect evidence to thesis more clearly.
	* Why: The current connection between evidence and thesis is weak and doesn't fully support the argument.
	* Reword: "Furthermore, requiring AI-use disclosure ensures that students are evaluated based on their own abilities rather than their access to technology."

---


## Part 3. Formatting Snippets and Elastic Context

Topic 6 emphasizes that snippet formatting depends on the host document. The same fact can be rendered as a **conversation transcript turn**, a report section, or a structured block.


### 3.1 Format the Same Evidence as Different Snippets


In [14]:
snippet_source = {
    "claim": "Require AI-use disclosure in written assignments.",
    "problem": "The thesis is hedged and does not take a firm stance.",
    "evidence_gap": "The survey example is mentioned but not interpreted.",
    "organization_gap": "Paragraph 2 jumps from fairness to privacy without a transition.",
}

conversation_snippet = dedent(f"""
Student: What is the biggest problem in my draft?
Writing Coach: Your thesis is currently weak because {snippet_source['problem']} Also, {snippet_source['evidence_gap']}
""").strip()

markdown_snippet = dedent(f"""
#### Draft Diagnosis
- Thesis: {snippet_source['problem']}
- Evidence: {snippet_source['evidence_gap']}
- Organization: {snippet_source['organization_gap']}
""").strip()

xml_snippet = dedent(f"""
<draft_diagnosis>
  <thesis>{snippet_source['problem']}</thesis>
  <evidence>{snippet_source['evidence_gap']}</evidence>
  <organization>{snippet_source['organization_gap']}</organization>
</draft_diagnosis>
""").strip()

for name, text in {
    "conversation": conversation_snippet,
    "markdown": markdown_snippet,
    "xml": xml_snippet,
}.items():
    print(f"===== {name} snippet =====")
    print(text)
    print("Approx tokens:", count_tokens(text))
    print()


===== conversation snippet =====
Student: What is the biggest problem in my draft?
Writing Coach: Your thesis is currently weak because The thesis is hedged and does not take a firm stance. Also, The survey example is mentioned but not interpreted.
Approx tokens: 44

===== markdown snippet =====
#### Draft Diagnosis
- Thesis: The thesis is hedged and does not take a firm stance.
- Evidence: The survey example is mentioned but not interpreted.
- Organization: Paragraph 2 jumps from fairness to privacy without a transition.
Approx tokens: 47

===== xml snippet =====
<draft_diagnosis>
  <thesis>The thesis is hedged and does not take a firm stance.</thesis>
  <evidence>The survey example is mentioned but not interpreted.</evidence>
  <organization>Paragraph 2 jumps from fairness to privacy without a transition.</organization>
</draft_diagnosis>
Approx tokens: 62



### 3.2 Brevity and Inertness

Good snippets are not only informative; they should also be brief and easy to combine without surprising tokenization effects.


In [15]:
verbose_snippet = dedent("""
The student's main claim is not yet sufficiently decisive, which creates uncertainty for the reader about whether the student is actually arguing for mandatory disclosure or merely thinking aloud about it.
""").strip()

brief_snippet = "Thesis is hedged; make the claim explicit and arguable."

print("Verbose snippet tokens:", count_tokens(verbose_snippet))
print("Brief snippet tokens:", count_tokens(brief_snippet))

if not USING_TIKTOKEN:
    print("\nUsing regex fallback, so the tokenization examples below are only approximate.")

for left, right in [("be", "am"), ("cat", "tail")]:
    merged = left + right
    newline_separated = left + "\n" + right
    print(f"\nPair: {left!r} + {right!r}")
    print("separate counts:", count_tokens(left) + count_tokens(right))
    print("merged string:", count_tokens(merged))
    print("newline separated:", count_tokens(newline_separated))


Verbose snippet tokens: 34
Brief snippet tokens: 14

Pair: 'be' + 'am'
separate counts: 2
merged string: 1
newline separated: 3

Pair: 'cat' + 'tail'
separate counts: 2
merged string: 3
newline separated: 3


### 3.3 Elastic Snippets

Topic 6 recommends preparing multiple sizes for the same source so the prompt can adapt to remaining budget.


In [16]:
elastic_versions = {
    "short": "Quoted problem: 'schools should think about disclosure maybe.' Diagnosis: the thesis is vague.",
    "medium": "Quoted thesis: 'AI can help students a lot ... schools should think about disclosure maybe.' Diagnosis: the thesis is vague, the evidence is under-explained, and paragraph 2 needs a transition.",
    "long": dedent(f"""
    Draft excerpt: {writing_case['draft_excerpt']}
    Teacher feedback:
    - {writing_case['teacher_feedback'][0]}
    - {writing_case['teacher_feedback'][1]}
    - {writing_case['teacher_feedback'][2]}
    Additional constraint: the student wants fixes that can be applied tonight.
    """).strip(),
}


def best_fitting_variant(variants, remaining_budget):
    feasible = []
    for name, text in variants.items():
        tokens = count_tokens(text)
        if tokens <= remaining_budget:
            feasible.append((tokens, name, text))
    if not feasible:
        return None, None, None
    tokens, name, text = max(feasible, key=lambda item: item[0])
    return name, text, tokens


for budget in [30, 60, 120]:
    name, text, tokens = best_fitting_variant(elastic_versions, budget)
    print(f"Budget {budget}: {name} ({tokens} tokens)")


Budget 30: short (19 tokens)
Budget 60: medium (42 tokens)
Budget 120: long (84 tokens)


### <b style="color:orange"> Task 2: Choose the best elastic snippet under a budget </b>

Assume this source is high-impact and dependency-free. Select the largest useful version that still fits the remaining budget.


In [26]:
remaining_budget = 60

# TODO: You should write code to select the best variant given the remaining budget. Hint: you can use the best_fitting_variant function defined above.
name, text, tokens = best_fitting_variant(elastic_versions, remaining_budget)
print("Remaining budget:", remaining_budget)
if name is None:
    print("No variant fits the remaining budget.")
else:
    print(f"Best variant: {name} ({tokens} tokens)")
    print(text)


Remaining budget: 60
Best variant: medium (42 tokens)
Quoted thesis: 'AI can help students a lot ... schools should think about disclosure maybe.' Diagnosis: the thesis is vague, the evidence is under-explained, and paragraph 2 needs a transition.


---


## Part 4. Relationships Among Prompt Elements

Prompt elements should be treated as data objects with at least three relationship dimensions from Topic 6:
1. **Position**: where the element belongs in the final order.
2. **Importance**: how painful it is to drop the element.
3. **Dependency**: what must co-exist or must not co-exist.


### 4.1 Represent Prompt Elements as Data


In [17]:
def make_element(element_id, text, position, importance, requires=None, excludes=None):
    return {
        "id": element_id,
        "text": text.strip(),
        "position": position,
        "importance": importance,
        "requires": requires or [],
        "excludes": excludes or [],
        "tokens": count_tokens(text.strip()),
    }


elements = [
    make_element("intro", "You are an HKBU academic writing coach writing a concise revision memo.", 0, 10),
    make_element("scope", "Focus on thesis, evidence, and organization. Ignore grammar-only edits unless meaning changes.", 10, 9),
    make_element("student_profile", f"Student profile: {writing_case['student_profile']}", 20, 6),
    make_element("assignment", f"Assignment: {writing_case['assignment']}", 30, 7),
    make_element("rubric", "Rubric: " + "; ".join(writing_case["rubric"]), 40, 7),
    make_element("teacher_feedback", "Teacher feedback: " + "; ".join(writing_case["teacher_feedback"]), 50, 8),
    make_element("draft_short", elastic_versions["short"], 60, 5, excludes=["draft_medium", "draft_long"]),
    make_element("draft_medium", elastic_versions["medium"], 60, 7, excludes=["draft_short", "draft_long"]),
    make_element("draft_long", elastic_versions["long"], 60, 9, excludes=["draft_short", "draft_medium"]),
    make_element(
        "output_format",
        "Output exactly 3 numbered revisions. Each item must include why it matters and one concrete rewrite suggestion.",
        90,
        9,
    ),
    make_element(
        "fewshot_example",
        'Example good feedback item: "1. Strengthen the thesis. Why: readers need a firm claim. Rewrite: Replace hedged wording with a direct claim."',
        95,
        5,
        requires=["output_format"],
    ),
    make_element(
        "refocus",
        "Now write the highest-leverage revision memo for this student.",
        100,
        10,
        requires=["output_format"],
    ),
    make_element(
        "transition",
        "# Revision Memo\n1.",
        110,
        10,
        requires=["output_format"],
    ),
]

for element in elements:
    print(element["id"], "tokens=", element["tokens"], "importance=", element["importance"], "requires=", element["requires"], "excludes=", element["excludes"])


intro tokens= 14 importance= 10 requires= [] excludes= []
scope tokens= 17 importance= 9 requires= [] excludes= []
student_profile tokens= 29 importance= 6 requires= [] excludes= []
assignment tokens= 25 importance= 7 requires= [] excludes= []
rubric tokens= 28 importance= 7 requires= [] excludes= []
teacher_feedback tokens= 35 importance= 8 requires= [] excludes= []
draft_short tokens= 19 importance= 5 requires= [] excludes= ['draft_medium', 'draft_long']
draft_medium tokens= 42 importance= 7 requires= [] excludes= ['draft_short', 'draft_long']
draft_long tokens= 84 importance= 9 requires= [] excludes= ['draft_short', 'draft_medium']
output_format tokens= 20 importance= 9 requires= [] excludes= []
fewshot_example tokens= 32 importance= 5 requires= ['output_format'] excludes= []
refocus tokens= 12 importance= 10 requires= ['output_format'] excludes= []
transition tokens= 6 importance= 10 requires= ['output_format'] excludes= []


### 4.2 Minimal Prompt Crafter

Follow the slide pseudocode directly:
1. Order elements by position.
2. Keep as many elements near the end as fit the budget.
3. Do not use sophisticated scoring.


In [18]:
def minimal_prompt_crafter(elements, budget):
    # Sort all elements from the beginning of the prompt to the end.
    ordered = sorted(elements, key=lambda element: element["position"])
    # Start with an empty list of selected elements.
    selected = []
    # Start the used token count at zero.
    used_tokens = 0

    # Walk backward so we try to keep elements near the end first.
    for element in reversed(ordered):
        # Check whether this element still fits inside the token budget.
        if used_tokens + element["tokens"] <= budget:
            # Keep this element.
            selected.append(element)
            # Update the number of tokens already used.
            used_tokens += element["tokens"]

    # Put the kept elements back into normal prompt order.
    selected = sorted(selected, key=lambda element: element["position"])
    # Combine the selected element texts into one prompt string.
    prompt = "\n\n".join(element["text"] for element in selected)
    # Return the selected elements, the final prompt, and the token count.
    return selected, prompt, used_tokens


### 4.2.1 Usage Example: Minimal Prompt Crafter

Given a token budget, the minimal crafter keeps as many **end-of-prompt** elements as possible. Let us test it with one shared example budget.


In [19]:
def show_selection(label, selected, used_tokens):
    print(f"{label} selected IDs:", [element["id"] for element in selected], f"({used_tokens} tokens)")
    print("\n\n")
    print(f"{label} selected snippets:")
    for element in selected:
        print(f"- {element['id']}: {element['text']}")


example_budget = 140
minimal_example_selected, minimal_example_prompt, minimal_example_tokens = minimal_prompt_crafter(elements, example_budget)

show_selection("Minimal crafter", minimal_example_selected, minimal_example_tokens)


Minimal crafter selected IDs: ['draft_medium', 'draft_short', 'output_format', 'fewshot_example', 'refocus', 'transition'] (131 tokens)



Minimal crafter selected snippets:
- draft_medium: Quoted thesis: 'AI can help students a lot ... schools should think about disclosure maybe.' Diagnosis: the thesis is vague, the evidence is under-explained, and paragraph 2 needs a transition.
- draft_short: Quoted problem: 'schools should think about disclosure maybe.' Diagnosis: the thesis is vague.
- output_format: Output exactly 3 numbered revisions. Each item must include why it matters and one concrete rewrite suggestion.
- fewshot_example: Example good feedback item: "1. Strengthen the thesis. Why: readers need a firm claim. Rewrite: Replace hedged wording with a direct claim."
- refocus: Now write the highest-leverage revision memo for this student.
- transition: # Revision Memo
1.


In [20]:
print("\nMinimal prompt:\n")
print(minimal_example_prompt)


Minimal prompt:

Quoted thesis: 'AI can help students a lot ... schools should think about disclosure maybe.' Diagnosis: the thesis is vague, the evidence is under-explained, and paragraph 2 needs a transition.

Quoted problem: 'schools should think about disclosure maybe.' Diagnosis: the thesis is vague.

Output exactly 3 numbered revisions. Each item must include why it matters and one concrete rewrite suggestion.

Example good feedback item: "1. Strengthen the thesis. Why: readers need a firm claim. Rewrite: Replace hedged wording with a direct claim."

Now write the highest-leverage revision memo for this student.

# Revision Memo
1.


### 4.3 Additive Greedy Prompt Crafter

Follow the slide pseudocode directly:
1. Start with an empty prompt.
2. Repeatedly choose the **highest-value feasible element**.
3. Add it, then sort the selected elements by position.


In [21]:
def selected_token_count(selected):
    # Add up the token count of every currently selected element.
    return sum(element["tokens"] for element in selected)



def can_add_element(element, selected, budget):
    # Collect the IDs of the elements that are already selected.
    selected_ids = {item["id"] for item in selected}

    # Reject the element if it has already been selected.
    if element["id"] in selected_ids:
        return False
    # Reject the element if adding it would exceed the budget.
    if selected_token_count(selected) + element["tokens"] > budget:
        return False
    # Reject the element if any required element is still missing.
    for required_id in element["requires"]:
        if required_id not in selected_ids:
            return False
    # Reject the element if any excluded element is already present.
    for excluded_id in element["excludes"]:
        if excluded_id in selected_ids:
            return False
    # If none of the checks failed, the element is feasible.
    return True



def greedy_prompt_crafter(elements, budget):
    # Start with an empty prompt.
    selected = []

    # Keep adding elements until no feasible element remains.
    while True:
        # Start a fresh list of feasible elements for this round.
        feasible_elements = []
        # Check every candidate element.
        for element in elements:
            # Keep only the elements that can be added now.
            if can_add_element(element, selected, budget):
                feasible_elements.append(element)

        # Stop when nothing else can be added.
        if not feasible_elements:
            break

        # Choose the feasible element with the highest importance value.
        best_element = max(feasible_elements, key=lambda element: element["importance"])
        # Add that best element to the selection.
        selected.append(best_element)
        # Re-sort the selected elements into normal prompt order.
        selected = sorted(selected, key=lambda element: element["position"])

    # Combine the selected element texts into one prompt string.
    prompt = "\n\n".join(element["text"] for element in selected)
    # Return the selected elements, the final prompt, and the token count.
    return selected, prompt, selected_token_count(selected)


### 4.3.1 Usage Example: Additive Greedy Prompt Crafter

Now apply the additive greedy crafter to the **same** `example_budget` and compare which snippets survive.


In [22]:
greedy_example_selected, greedy_example_prompt, greedy_example_tokens = greedy_prompt_crafter(elements, example_budget)

show_selection("Additive greedy crafter", greedy_example_selected, greedy_example_tokens)


Additive greedy crafter selected IDs: ['intro', 'scope', 'draft_long', 'output_format'] (135 tokens)



Additive greedy crafter selected snippets:
- intro: You are an HKBU academic writing coach writing a concise revision memo.
- scope: Focus on thesis, evidence, and organization. Ignore grammar-only edits unless meaning changes.
- draft_long: Draft excerpt: AI can help students a lot and sometimes it is not fair because some people know how to use it better, so schools should think about disclosure maybe.
Teacher feedback:
- The thesis is currently vague and hedged.
- The evidence is relevant, but the explanation is thin.
- Paragraph 2 jumps from fairness to privacy without a transition.
Additional constraint: the student wants fixes that can be applied tonight.
- output_format: Output exactly 3 numbered revisions. Each item must include why it matters and one concrete rewrite suggestion.


In [23]:
print("\nAdditive greedy prompt:\n")
print(greedy_example_prompt)


Additive greedy prompt:

You are an HKBU academic writing coach writing a concise revision memo.

Focus on thesis, evidence, and organization. Ignore grammar-only edits unless meaning changes.

Draft excerpt: AI can help students a lot and sometimes it is not fair because some people know how to use it better, so schools should think about disclosure maybe.
Teacher feedback:
- The thesis is currently vague and hedged.
- The evidence is relevant, but the explanation is thin.
- Paragraph 2 jumps from fairness to privacy without a transition.
Additional constraint: the student wants fixes that can be applied tonight.

Output exactly 3 numbered revisions. Each item must include why it matters and one concrete rewrite suggestion.


### <b style="color:orange"> Task 3: Implement the subtractive greedy crafter </b>

Fill in the blanks inside `subtractive_greedy_prompt_crafter()` by following the slide pseudocode:
1. Start with all elements.
2. While the prompt is over budget or infeasible, remove the **lowest-value** removable element.
3. Prune any elements whose requirements break.

You can use these two helper functions directly:
- `prompt_is_feasible(selected, budget)`: checks whether the current prompt both fits the token budget and satisfies all requirement/exclusion rules.
- `prune_broken_requirements(selected)`: removes elements whose required dependencies are no longer present after an earlier removal.

After filling in the blanks, test the crafter with the same `example_budget` and inspect which snippets remain.


In [25]:
def prompt_is_feasible(selected, budget):
    # Collect the IDs of all currently selected elements.
    selected_ids = {element["id"] for element in selected}

    # Fail immediately if the prompt is longer than the budget.
    if selected_token_count(selected) > budget:
        return False

    # Check every selected element one by one.
    for element in selected:
        # Fail if this element is missing one of its required elements.
        for required_id in element["requires"]:
            if required_id not in selected_ids:
                return False
        # Fail if this element conflicts with another selected element.
        for excluded_id in element["excludes"]:
            if excluded_id in selected_ids:
                return False

    # If no rule is broken, the prompt is feasible.
    return True



def prune_broken_requirements(selected):
    # Keep removing elements whose requirements are no longer satisfied.
    while True:
        # Collect the IDs that are still present.
        selected_ids = {element["id"] for element in selected}
        # Start a list for elements whose requirements are now broken.
        broken = []

        # Check each selected element.
        for element in selected:
            # Track whether this element is missing any required dependency.
            missing_requirement = False
            # Look through every required element ID.
            for required_id in element["requires"]:
                # Mark this element as broken if a requirement is missing.
                if required_id not in selected_ids:
                    missing_requirement = True
                    break
            # Save this element if its requirements are broken.
            if missing_requirement:
                broken.append(element)

        # Stop when there are no more broken elements.
        if not broken:
            return selected

        # Collect the IDs of all broken elements.
        broken_ids = {element["id"] for element in broken}
        # Remove all broken elements in one pass.
        selected = [element for element in selected if element["id"] not in broken_ids]



def subtractive_greedy_prompt_crafter(elements, budget):
    # Start with all elements, already ordered like the final prompt.
    selected = sorted(list(elements), key=lambda element: element["position"])

    # TODO: You should write code to check whether the current selection is feasible, and if not, remove the lowest-importance element and prune any elements whose requirements are now broken. Repeat until the selection is feasible.
    while not prompt_is_feasible(selected, budget):
        if not selected:
            break
        worst= min(selected, key=lambda e:(e["importance"],-e["tokens"]))
        selected = [e for e in selected if e["id"] != worst["id"]]
        selected = prune_broken_requirements(selected)
        selected = sorted(selected, key=lambda element: element["position"])



    # Combine the remaining element texts into one prompt string.
    prompt = "\n\n".join(element["text"] for element in selected)
    # Return the selected elements, the final prompt, and the token count.
    return selected, prompt, selected_token_count(selected)


# Run the subtractive greedy crafter on the shared example budget.
subtractive_example_selected, subtractive_example_prompt, subtractive_example_tokens = subtractive_greedy_prompt_crafter(elements, example_budget)

# Show which elements the subtractive greedy crafter kept.
show_selection("Subtractive greedy crafter", subtractive_example_selected, subtractive_example_tokens)


Subtractive greedy crafter selected IDs: ['intro', 'scope', 'output_format', 'refocus', 'transition'] (69 tokens)



Subtractive greedy crafter selected snippets:
- intro: You are an HKBU academic writing coach writing a concise revision memo.
- scope: Focus on thesis, evidence, and organization. Ignore grammar-only edits unless meaning changes.
- output_format: Output exactly 3 numbered revisions. Each item must include why it matters and one concrete rewrite suggestion.
- refocus: Now write the highest-leverage revision memo for this student.
- transition: # Revision Memo
1.


## Pitfalls and Extension

**Common pitfall:** treating prompt assembly as simple string concatenation. This often buries key instructions in the middle, wastes budget on low-value context, and gives a raw completion model a weak ending cue.
- Fix: represent prompt elements explicitly and track position, importance, dependencies, exclusions, and the final transition cue.

**Optional extension:** compare the minimal, additive greedy, and subtractive greedy crafters across several budgets and inspect when their selections begin to differ.


## Key Takeaways

1. A strong prompt usually needs an **introduction, context, refocus, and transition**.
2. **Advice conversations, Markdown reports, and structured documents** can all be implemented as raw completion prompts.
3. Good snippets are **modular, natural, brief, and token-aware**.
4. Elastic snippets let you trade off context richness against token budget.
5. **Minimal, additive greedy, and subtractive greedy** crafters make different trade-offs when selecting prompt elements.
6. With `ollama.generate()`, the ending of the prompt matters even more because the model is asked to continue the text directly.


---


### <b style="color:orange"> Submission </b>
Make sure that you **1) successfully generate all outputs in this notebook and 2) finish all the tasks**.

Submit the updated notebook with the filename ***YourName_YourStudentID.ipynb*** to the Moodle.
